# Peterborough Restaurant Reviews — Market EDA
**Phase 3 / Step 5 — Exploratory Analysis**

All charts are interactive (Plotly). Hover for values, click legend to toggle series.

> **Data note**: Results reflect the *observed sample* of Google Reviews collected for 25 Peterborough restaurants. This is not a census of all customers or all reviews. See Section 5A for population definitions and coverage.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.stats as stats

import os as _os
PARQUET = _os.path.join(_os.environ.get("PTBO_BASE", ".."), "02. Outputs_Data Mining", "reviews_analysis_ready.parquet")
RECENT_MONTHS = 12
MIN_REVIEWS_RANK = 30
MIN_REVIEWS_TREND = 10
PEER_ORDER = ["Fast Casual & Counter Service", "Breakfast, Diner & Family Casual", "Casual Full-Service Ethnic Dining", "Pub, Burger & Smokehouse Casual", "Premium Full-Service & Experience Dining"]

df = pd.read_parquet(PARQUET)
for col in ["published_at","scraped_at","owner_response_at"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")
df["pub_ym"] = df["published_at"].dt.to_period("M")
df["pub_month_num"] = df["published_at"].dt.month
df["_pop_rating"] = df["rating_analysis_eligible"].fillna(0) == 1
df["_pop_nlp"]    = df["nlp_eligible"].fillna(0) == 1
rp = df[df["_pop_rating"]].copy()
np_ = df[df["_pop_nlp"]].copy()
print(f"Rating pop (A): {df['_pop_rating'].sum():,} | NLP pop (B): {df['_pop_nlp'].sum():,}")

## 5A — Population and Data Coverage

In [ ]:
# Population funnel
funnel = dict(
    labels=["All reviews","Rating pop. A","Text reviews","NLP pop. B","Gold eligible"],
    values=[len(df), int(df["_pop_rating"].sum()), int(df["has_text"].fillna(0).sum()),
            int(df["_pop_nlp"].sum()), int(df["gold_annotation_eligible"].fillna(0).sum())]
)
fig = go.Figure(go.Funnel(y=funnel["labels"], x=funnel["values"],
    textinfo="value+percent initial",
    marker=dict(color=["#B5D4F4","#85B7EB","#378ADD","#1D9E75","#0F6E56"])))
fig.update_layout(title="Review population funnel", height=400)
fig.show()

In [ ]:
# Review volume per restaurant
vol = rp.groupby("restaurant_name")["review_id"].count().reset_index()
vol.columns = ["restaurant_name","n_reviews"]
vol = vol.sort_values("n_reviews", ascending=False)
vol["cumshare"] = vol["n_reviews"].cumsum() / vol["n_reviews"].sum()
fig = px.bar(vol, x="restaurant_name", y="n_reviews",
    title="Review volume per restaurant (Population A)",
    color_discrete_sequence=["#378ADD"],
    labels={"n_reviews":"Reviews","restaurant_name":"Restaurant"})
fig.add_hline(y=MIN_REVIEWS_RANK, line_dash="dot", line_color="#D85A30",
              annotation_text=f"Min rank threshold ({MIN_REVIEWS_RANK})")
fig.update_xaxes(tickangle=45)
fig.show()
print(f"Top-5 concentration: {vol.head(5)['n_reviews'].sum()/vol['n_reviews'].sum():.1%}")

## 5B — Market-Level Rating Analysis

In [ ]:
# Star distribution + RW vs RestW
star_dist = rp["review_stars"].value_counts().sort_index().reset_index()
star_dist.columns = ["stars","count"]
star_dist["pct"] = star_dist["count"] / star_dist["count"].sum() * 100

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    "Star distribution", "Review-weighted vs Restaurant-weighted mean"])
fig.add_trace(go.Bar(x=star_dist["stars"], y=star_dist["count"],
    marker_color=["#D85A30","#F0997B","#EF9F27","#9FE1CB","#1D9E75"],
    text=star_dist["pct"].round(1).astype(str)+"%", textposition="outside",
    name="Count"), row=1, col=1)

rw_m = rp["review_stars"].mean()
restw_m = rp.groupby("restaurant_id")["review_stars"].mean().mean()
fig.add_trace(go.Bar(x=["Review-weighted","Restaurant-weighted"],
    y=[rw_m, restw_m],
    marker_color=["#1D9E75","#7F77DD"],
    text=[f"{rw_m:.3f}", f"{restw_m:.3f}"], textposition="outside",
    name="Mean"), row=1, col=2)
fig.update_layout(title="Market rating overview", height=450, showlegend=False)
fig.show()
print(f"Difference (RW − RestW): {rw_m - restw_m:+.3f} stars")

## 5C — Temporal Analysis

In [ ]:
# Monthly volume + rating trend
monthly = (
    rp.groupby(rp["pub_ym"].astype(str))
    .agg(n=("review_id","count"), mean_stars=("review_stars","mean"))
    .reset_index().rename(columns={"pub_ym":"ym"})
)
monthly["rolling_mean"] = monthly["mean_stars"].rolling(3, min_periods=1).mean()
monthly["plot"] = monthly["n"] >= MIN_REVIEWS_TREND

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    subplot_titles=["Monthly review volume","Monthly mean rating (review-weighted)"])
fig.add_trace(go.Bar(x=monthly["ym"], y=monthly["n"],
    marker_color="#B5D4F4", name="Volume"), row=1, col=1)
plotable = monthly[monthly["plot"]]
fig.add_trace(go.Scatter(x=plotable["ym"], y=plotable["mean_stars"],
    mode="markers+lines", line=dict(dash="dot", color="#378ADD"),
    name="Monthly avg"), row=2, col=1)
fig.add_trace(go.Scatter(x=plotable["ym"], y=plotable["rolling_mean"],
    mode="lines", line=dict(color="#D85A30", width=2.5),
    name="3M rolling avg"), row=2, col=1)
fig.update_layout(height=600, title="Review volume and rating trend")
fig.show()

## 5D — Restaurant Benchmarking

In [ ]:
# Rating vs volume scatter
bench = rp.groupby(["restaurant_id","restaurant_name","operational_peer_group"]).agg(
    n=("review_id","count"), mean_stars=("review_stars","mean")).reset_index()
bench["low_sample"] = bench["n"] < MIN_REVIEWS_RANK
bench["symbol"] = bench["low_sample"].map({True:"triangle-up", False:"circle"})

fig = px.scatter(bench, x="n", y="mean_stars",
    color="operational_peer_group",
    symbol="symbol",
    text="restaurant_name",
    hover_data={"n":True,"mean_stars":":.3f","low_sample":True},
    title="Rating vs review volume — restaurant benchmarking",
    labels={"n":"Number of reviews","mean_stars":"Mean star rating",
            "operational_peer_group":"Peer group"})
fig.add_hline(y=rp["review_stars"].mean(), line_dash="dash", line_color="gray",
    annotation_text=f"Market avg {rp['review_stars'].mean():.2f}★")
fig.update_traces(textposition="top center", textfont_size=8)
fig.update_layout(height=550)
fig.show()

## 5E — Peer-Group Benchmarking

In [ ]:
# Peer group rating comparison
peer = rp.groupby("operational_peer_group").agg(
    rw_mean=("review_stars","mean"),
    n=("review_id","count"),
    n_rest=("restaurant_id","nunique")).reset_index()
restw = rp.groupby(["operational_peer_group","restaurant_id"])["review_stars"].mean()    .groupby(level=0).mean().reset_index().rename(columns={"review_stars":"restw_mean"})
peer = peer.merge(restw, on="operational_peer_group")

fig = go.Figure()
fig.add_trace(go.Bar(name="Review-weighted", x=peer["operational_peer_group"],
    y=peer["rw_mean"], marker_color="#1D9E75",
    text=peer["rw_mean"].round(3), textposition="outside"))
fig.add_trace(go.Bar(name="Restaurant-weighted", x=peer["operational_peer_group"],
    y=peer["restw_mean"], marker_color="#7F77DD",
    text=peer["restw_mean"].round(3), textposition="outside"))
fig.update_layout(barmode="group", title="Peer-group mean rating — RW vs RestW",
    yaxis=dict(range=[3.5, 5.3]), height=450)
fig.show()

## 5F — Review and Engagement Behaviour

In [ ]:
# Review length by star + owner response rate
length = np_.groupby("review_stars")["text_length_words"].median().reset_index()
resp   = rp.groupby("review_stars")["has_owner_response"].mean().reset_index()
resp["pct"] = resp["has_owner_response"] * 100

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    "Median review length by star (NLP pop.)",
    "Owner response rate by star (Rating pop.)"])
fig.add_trace(go.Bar(x=length["review_stars"], y=length["text_length_words"],
    marker_color=["#D85A30","#F0997B","#EF9F27","#9FE1CB","#1D9E75"],
    text=length["text_length_words"].round(1), textposition="outside",
    name="Median words"), row=1, col=1)
fig.add_trace(go.Bar(x=resp["review_stars"], y=resp["pct"],
    marker_color=["#D85A30","#F0997B","#EF9F27","#9FE1CB","#1D9E75"],
    text=resp["pct"].round(1).astype(str)+"%", textposition="outside",
    name="Response rate %"), row=1, col=2)
fig.update_layout(height=420, showlegend=False,
    title="Review behaviour and owner engagement")
fig.show()

## 5G — Structured Context (coverage-gated)

In [ ]:
# Detailed ratings vs overall star
for col, label in [("detailed_rating_food","Food"),
                   ("detailed_rating_service","Service"),
                   ("detailed_rating_atmosphere","Atmosphere")]:
    if col not in rp.columns: continue
    cov = rp[col].notna().mean()
    if cov < 0.15:
        print(f"{label}: coverage {cov:.1%} — skipped")
        continue
    agg = rp.groupby("review_stars")[col].mean().reset_index()
    corr = rp[[col,"review_stars"]].dropna().corr().iloc[0,1]
    fig = px.bar(agg, x="review_stars", y=col,
        title=f"{label} detailed rating by overall star (r={corr:.2f}, coverage={cov:.0%})",
        color_discrete_sequence=["#378ADD"],
        labels={"review_stars":"Star rating", col:f"Mean {label} rating"})
    fig.show()